# RAG Day 4

## Evaluation!

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Keep in mind how you would evaluate RAG for your business</h2>
            <span style="color:#181;">This is such an important part of building an accurate and reliable RAG pipeline. And it's applicable to many aspects of solving business problems with LLMs. People are often focused on RAG architecture and RAG frameworks for their business. But even more important: evaluations!</span>
        </td>
    </tr>
</table>

### Fisrt step:
- Curate a test set: examples question set with the right context identified and reference answers provided:
    - can be provided by questions made by costumer in the Q&A session of your website
    - constantly add questions while you find issues in your llm
    - Example:
        ```
        {
            "question": "Who won the prestigious IIOTY award in 2023?", "keywords": ["Maxine", "Thompson", "IIOTY"], 
            "reference_answer": "Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.", "category": "direct_fact"
        }
        ```
- Types of tests:
    - how effective is your retrieval:
        - MRR (Mean Reciprocal Rank) - if you very first chunc have the information you are looking for - the relevant context
            - Average inverse rank of first hit; if the first chunk always has the relevant context
        - nDGG (Normalized Discounted Cumulative Gain) - measure how godd are you surfacing the most important context to the very top
            - Did relevant chunks get ranked higher up
            - the chunks that have relevant contxt is always on top
        - Recall@K: Recall@3, or 5, or 6 - Proportion of tests where the relevant context was in the top K chunks - the top 3 chunks is in the relevant context
            - of if you have multiple keywords to look for, <b>keyword coverage</b> is similar recall metric
        - Precision@K: proportion of the chunks that are relevant
            - For Precision@5: how much of the 5 chunks have relevant context: 50% surfacing relevant contents means only a half of 5 have relevant content
            - waist of time with irrelevant content, affect precision

- Measure answers:
    - Use LLM-as-a-judge to score provided answers against criteria like accuray, completeness and relevance
    - Example: give the expected answer and the real answer and ask to LLM if it as good answer, give a score 1-5, 
    - how complete, how accurate, how relevant are question that can be made?
    




In [5]:
from evaluation import test

In [6]:
tests = test.load_tests()

In [7]:
len(tests)

150

In [8]:
example = tests[0]
print(example.question)
print(example.category)
print(example.reference_answer)
print(example.keywords)


Who won the prestigious IIOTY award in 2023?
direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


In [9]:
# count the categories inside tests
from collections import Counter
count = Counter([t.category for t in tests])
count

Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [10]:
from evaluation.eval import evaluate_retrieval, evaluate_answer

In [11]:
evaluate_retrieval(example)

RetrievalEval(mrr=0.6666666666666666, ndcg=0.6399069297160626, keywords_found=2, total_keywords=3, keyword_coverage=66.66666666666666)

In [12]:
eval, answer, chunks = evaluate_answer(example)

In [13]:
eval

AnswerEval(feedback="The generated answer correctly identifies Maxine as the winner and mentions her role and company, which adds context not present in the reference, but its main flaw is that it states 'Maxine, the Senior Data Engineer at Insurellm' instead of the full name 'Maxine Thompson.' This means it is not an exactly perfect match in naming, which impacts accuracy. The answer does address all details of the award with sufficient completeness and is highly relevant to the question.", accuracy=4.0, completeness=4.0, relevance=5.0)

In [14]:
print(eval.feedback)
print(eval.accuracy)
print(eval.completeness)
print(eval.relevance)

The generated answer correctly identifies Maxine as the winner and mentions her role and company, which adds context not present in the reference, but its main flaw is that it states 'Maxine, the Senior Data Engineer at Insurellm' instead of the full name 'Maxine Thompson.' This means it is not an exactly perfect match in naming, which impacts accuracy. The answer does address all details of the award with sufficient completeness and is highly relevant to the question.
4.0
4.0
5.0


run the bellow in the folder (.venv) PS D:\AI\ai-llm-whole-course\Week5> 

`python evaluator.py`

or

`py evaluator.py`

Changing the chunks size (ingest.py) and the RETRIEVAL_K (answer.py) we got these results
(don't forget to run py ingest.py after changes):
```
chunks= 500 / RETRIEVAL_K = 10
MRR 0.8819
DCG 0.8614
KCoverage 96
```

```
chunks= 1667 / RETRIEVAL_K = 3
MRR 0.8868
DCG 0.8942
KCoverage 93.1
```
